In [232]:
import jax.numpy as jnp
from jax import grad
from diffrax import diffeqsolve, Tsit5

#import sys
#sys.path.append('/Users/mortenpedersen/code/jaxgeometry/jaxgeometry')
# from src.jaxgeometry.manifolds.landmarks import *

Import kernel functions from jaxgeometry:


In [233]:
import sys
sys.path.append('/Users/mortenpedersen/code/jaxgeometry/jaxgeometry/src/')
from jaxgeometry.manifolds.landmarks import *

In [234]:
k = 3

M = landmarks(N = k,m=2, k_sigma=0.1*jnp.eye(2),k_alpha=1, kernel="Gaussian")

In [235]:
# jnp.linalg.norm(q[0][0:2] - q[0][0:2])

M.k(q[0][0:2])

Array(1., dtype=float32)

In [236]:
q[0].shape, M.gsharp(q).shape, p.shape

((6,), (6, 6), (3, 2))

In [237]:
# q = (jnp.array([1,0,1.5,0.5,1.5,0.5]),[0.])
# p = jnp.array([1,0,1.5,0.5,1.5,0.5])
# prod = jnp.tensordot(M.gsharp(q),p,(1,0))

# print('gsharp: ', M.gsharp(q))

# print('product: ', prod)
# print('product shape: ', prod.shape)

In [238]:
q_flat = jnp.array([1,0,1.5,0.5,2,0.5])
p_flat = jnp.array([1,0,1.5,0.5,2,0.5])
q = q_flat.reshape(3,2)
p = p_flat.reshape(3,2)

w = jnp.array([[0.2,0.8,0.8],[0.8,0.2,0.2]]) # weight-matrix (POU)

w[0]


Array([0.2, 0.8, 0.8], dtype=float32)

In [239]:
#### TODO: - inkorporér kernens afhængighed af j, - implementer grad_w og dk


# Define parameters
k = M.N  # Number of indices for q and p
r = 2  # Number of functions for w
m = M.m # ambient dimension

w = jnp.array([[0.2,0.8,0.8],[0.8,0.2,0.2]]) # weight-matrix (POU)

def K_sigma_j(q_l, q_i):
    return M.k(q_l - q_i) * jnp.eye(m)

def grad_w_j(q_l, t):
    # Define gradient of w_j function here
    return ...  # Replace with actual implementation

def nu(x, q, p, j):
    # q,p are full k*m dimensional state vectors, x is a single m-dimensional vector

    out = jnp.zeros(m)

    for i in range(k):
        out += K_sigma_j(x, q[i]) * w[j, i] * p[i]

    return out

# Define the system of ODEs
def ode_system(t, y):
    # y = [q_1, q_2, ..., q_k, p_1, p_2, ..., p_k, grad_w_1_1, .., grad_w_1_k, ..., grad_w_r_1, .., grad_w_r_k], where each of these elements is an m-dimensional vector
    q = y[:k*m].reshape((k, m))
    p = y[k*m:2*k*m].reshape((k, m))
    grad_w = y[2*k*m:].reshape((r,k,m))

    dq_dt = jnp.zeros(k,2)
    dp_dt = jnp.zeros(k,2)

    # Compute dq/dt
    for l in range(k):
        v_l = jnp.zeros(m)
        for j in range(r):
            v_l = w[j, l] * nu(q[l], q, p, j)     # TODO: compute nu as a matrix only once
        dq_dt = dq_dt.at[l].set(v_l)

    # Compute dp/dt
    for l in range(k):
        
        nu_l = lambda j : nu(q[l], q, p, j)

        term_dv_1 = jnp.zeros(m,m)
        term_dv_2 = jnp.zeros(m,m)
        term_nus = jnp.zeros(m)

        for j in range(r):
            term_nus += jnp.dot(p[l],nu_l(j)) * grad_w[j,l]
        
        for j in range(r):
            term_dv_1 += jnp.outer(grad_w[j,l], nu_l(j))
                                 
        for j in range(r):
            for i in range(k):
                term_dv_2 += w[j,l] * w[j,i] * dk[j,l,i] @ p[i]
        
        dp_dt = dp_dt.at[l].set((term_dv_1 + term_dv_1) @ p[l] + term_nus)
        

    # Combine derivatives into one array
    return jnp.concatenate((dq_dt.flatten(), dp_dt.flatten(), dw_dt.flatten()))

# Initial conditions
initial_conditions = jnp.zeros(2*k*m + r*k*m))  

# Time span for the solution
t_span = (0.0, 10.0)  # Adjust according to your problem
t_eval = jnp.linspace(t_span[0], t_span[1], 100)  # Evaluation points

# Set up the differential equation solver
solver = Tsit5()

# Solve the ODE system
solution = diffeqsolve(ode_system, solver, t_span, y0=initial_conditions, t_eval=t_eval)

# Extract results
q_sol = solution.ys[:k]  # q values
p_sol = solution.ys[k:k + k]  # p valuesw_sol = solution.ys[k + k:]  # w values

# The results can be processed further or plotted as needed


SyntaxError: unmatched ')' (2130190590.py, line 71)

In [270]:
# create weight-matrices (POU function at landmarks)

w = jnp.array([1,1,1]) #jnp.array([0.2,0.8,0.8]) # weight-matrix (POU)
w_1_diag = jnp.diag(jnp.repeat(w, 2))
w_2_diag = jnp.eye(len(w)*2) - w_1_diag

print(w_1_diag, '\n\n', w_2_diag)

[[1 0 0 0 0 0]
 [0 1 0 0 0 0]
 [0 0 1 0 0 0]
 [0 0 0 1 0 0]
 [0 0 0 0 1 0]
 [0 0 0 0 0 1]] 

 [[0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0. 0.]]


In [322]:
# create both kernels 

k = 3

sigma_1 = 0.1
sigma_2 = 0.5
M1 = landmarks(N = k,m=2, k_sigma=sigma_1*jnp.eye(2),k_alpha=1, kernel="Gaussian")
M2 = landmarks(N = k,m=2, k_sigma=sigma_2*jnp.eye(2),k_alpha=1, kernel="Gaussian")

M = landmarks(N = k,m=2, k_sigma=0*jnp.eye(2),k_alpha=1, kernel="Gaussian")  # k_sigma=0 since it is not used in the hamiltonian, which is overwritten below

from jaxgeometry.Riemannian import metric
metric.initialize(M)

kernel = lambda q1, q2: w_1_diag @ M1.K(q1, q2) @ w_1_diag + w_2_diag @ M2.K(q1, q2) @ w_2_diag # nb: this takes only coord as input 
sharp = lambda q: w_1_diag @ M1.gsharp(q) @ w_1_diag + w_2_diag @ M2.gsharp(q) @ w_2_diag # nb: this takes coord+chart as input

M.H = lambda q,p: 0.5*jnp.tensordot(p,jnp.tensordot(
                                    sharp(q),
                                    p,(1,0)),(0,0))

from jaxgeometry.dynamics import Hamiltonian
Hamiltonian.initialize(M)


In [311]:
q_flat = jnp.array([0.,0.,0.1,0.,0.2,0.])
p_flat = jnp.array([0.,0.1,0,0.1,0,0.1])

q = (q_flat, [0.])

# M.Exp_Hamiltonian(q,p_flat, n_steps=100)
n_steps_flow = 200
_dts = dts(n_steps=n_steps_flow)
(_,qps,charts_qp) = M.Hamiltonian_dynamics(q,p_flat,_dts)

qs = qps[:,0]



Plot results:

In [312]:
q_0 = q[0].reshape(k,2)
p_0 = p_flat.reshape(k,2)
n_landmark = k

minx = np.min(qs[:, ::2])
miny = np.min(qs[:, ::1])
maxx = np.max(qs[:, ::2])
maxy = np.max(qs[:, ::1])


# create grid 
pts_v_spatial=20
pts_h_spatial=20
K_spatial = pts_h_spatial*pts_v_spatial # number of evaluation points
x_spatial,y_spatial = np.meshgrid(np.linspace(minx,maxx,pts_v_spatial),np.linspace(miny,maxy,pts_h_spatial))
x_spatial = x_spatial.flatten(); y_spatial = y_spatial.flatten()
xy_spatial = jnp.vstack((x_spatial,y_spatial)).T


In [313]:
import matplotlib.pyplot as plt
import matplotlib.animation as animation

fig, ax = plt.subplots()

# Set up the plot
ax.set_xlim(minx, maxx)
ax.set_ylim(miny, maxy)
ax.set_aspect('equal')
ax.set_xlabel('x')
ax.set_ylabel('y')

# Initialize the scatter plot
scatter = ax.scatter([], [], color='red')

# Update function for the animation
def update(frame):
    # Get the x and y coordinates for the current frame
    x = qs[frame, ::2]
    y = qs[frame, 1::2]

    # Update the scatter plot
    scatter.set_offsets(np.column_stack((x, y)))

    return scatter,

# Create the animation
ani = animation.FuncAnimation(fig, update, frames=n_steps_flow, interval=20, blit=True)

# Show the animation
video = ani.to_html5_video()
html = display.HTML(video)
display.display(html)
plt.close()

In [314]:
### check results

w_1_diag @ M1.gsharp((q_flat,[0.])) @ w_1_diag  +  w_2_diag @ M2.gsharp((q_flat,[0.])) @ w_2_diag 


# one ham

a1 = jnp.tensordot(w_1_diag @ p_flat,
              jnp.tensordot(M1.gsharp((q_flat,[0.])), 
                            w_1_diag @ p_flat,(1,0)),(0,0))

a2 = jnp.tensordot(w_2_diag @ p_flat,
              jnp.tensordot(M2.gsharp((q_flat,[0.])), 
                            w_2_diag @ p_flat,(1,0)),(0,0))



b1 = jnp.tensordot(p_flat,
              jnp.tensordot(w_1_diag @ M1.gsharp((q_flat,[0.])) @ w_1_diag, 
                            p_flat,(1,0)),(0,0))

print(jnp.allclose(a1,b1))

# two hams

b2 =  jnp.tensordot(p_flat,
                    jnp.tensordot(w_1_diag @ M1.gsharp((q_flat,[0.])) @ w_1_diag + w_2_diag @ M2.gsharp((q_flat,[0.])) @ w_2_diag, 
                                  p_flat,(1,0)),(0,0))

print(jnp.allclose(b2, a1 + a2))


True
True


Advect grid along flow:

In [334]:
jnp.allclose(M1.K(q[0]+1,q[0]), kernel(q[0]+1,q[0]))

Array(True, dtype=bool)

In [333]:
n_landmark = k

minx = np.min(qs[:, ::2])
miny = np.min(qs[:, ::1])
maxx = np.max(qs[:, ::2])
maxy = np.max(qs[:, ::1])

#minx =-0.25
#miny =-1
#maxx =1.5
#maxy =0.2

pts_v =100
pts_h =50
K = pts_h*pts_v # number of evaluation points
x,y = np.meshgrid(np.linspace(minx,maxx,pts_v),np.linspace(miny,maxy,pts_h))
x = x.flatten(); y = y.flatten()
xy = jnp.vstack((x,y)).T

# flow arbitrary points of N
def ode_Hamiltonian_advect(c,y):
    t,x,chart = c
    qp, = y
    q = qp[0]
    p = qp[1]

    # jax.debug.print("{x}", x=x.shape)
    # jax.debug.print("{q}", q=q.shape)
    # jax.debug.print("{MK}", MK=M.K(x,q).shape)

    dxt = jnp.tensordot(kernel(x,q),p,(1,0)).reshape((-1,M.m))
    # dxt = jnp.tensordot(M1.K(x,q),p,(1,0)).reshape((-1,M.m))
    return dxt

# backwards flow arbitrary points 
def ode_Hamiltonian_advect_rev(c,y):
    t,x,chart = c
    qp, = y
    q = qp[0]
    p = qp[1]

    # dxt = -jnp.tensordot(kernel(x,q),p,(1,0)).reshape((-1,M.m))
    dxt = -jnp.tensordot(M1.K(x,q),p,(1,0)).reshape((-1,M.m))
    return dxt

M.Hamiltonian_advect = lambda xs,qps,dts: integrate(ode_Hamiltonian_advect,
                                                    None,
                                                    xs[0].reshape((-1,M.m)),
                                                    xs[1],
                                                    dts,
                                                    qps[::1])

M.Hamiltonian_advect_rev = lambda xs,qps,dts: integrate(ode_Hamiltonian_advect_rev,
                                                        None,
                                                        xs[0].reshape((-1,M.m)),
                                                        xs[1],
                                                        dts,
                                                        qps[::-1])

# grid/ambient flow
_,xs = M.Hamiltonian_advect((xy.flatten(),M.chart()),qps,_dts)

# flow of landmark shape (for correctness check)
_,shape_forward = M.Hamiltonian_advect((q[0],M.chart()),qps,_dts)

print(qps.shape, xs.shape)
qs = qps[:,0]
ms = qps[:,1]
#M.plot_path(zip(qs,charts),v,linewidth=1.5)


                                               

TypeError: dot_general requires contracting dimensions to have the same shape, got (6,) and (10000,).